# Galileo Brain v6 "Onnipotenza" — Continual Training

Fine-tuning continuo del modello v5 (Qwen3-4B + LoRA) con il dataset v6 stratificato:
- **20,319 training** examples (4 strati: replay + direct action + context + adversarial)
- **108 eval** examples (bilanciati per intent)

Target: intent accuracy >=95%, 0 parse errors, 42/42 azioni coperte

In [ ]:
# Cell 1: Installazione Unsloth + dipendenze
import sys
if "google.colab" in sys.modules:
    !pip install --no-deps trl peft accelerate bitsandbytes
    !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
    !pip install --no-deps xformers triton
    !pip install transformers==4.56.2 trl==0.22.2 datasets

print("\u2705 Installazione completata!")
print("\u26a0\ufe0f  ORA RIAVVIA IL RUNTIME: Runtime \u2192 Riavvia runtime")
print("   Poi esegui dalla Cell 2 in poi")

In [ ]:
# Cell 2: Caricamento modello Qwen3-4B con Unsloth
# NOTA: Carichiamo il modello BASE (non il v5) perche' Unsloth applica LoRA fresco.
# Il v5 aveva gia' converguto — ripartire dal base con il dataset v6 completo
# (che include il replay layer) e' piu' pulito del continual training.
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

print(f"\u2705 Modello caricato! Device: {model.device}")
print(f"   VRAM usata: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

In [ ]:
# Cell 3: Configurazione LoRA adapter
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\u2705 LoRA configurato!")
print(f"   Parametri trainabili: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
print(f"   VRAM usata: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

In [ ]:
# Cell 4: Upload e caricamento dataset
from google.colab import files
import json, glob

print("\ud83d\udcc1 Carica i 2 file JSONL:")
print("   1) galileo-brain-v6.jsonl          (train ~20K)")
print("   2) galileo-brain-v6-eval.jsonl     (eval ~108)")
uploaded = files.upload()

train_file = [f for f in uploaded if 'eval' not in f and f.endswith('.jsonl')][0]
eval_file  = [f for f in uploaded if 'eval' in f and f.endswith('.jsonl')][0]

for label, fname in [("TRAIN", train_file), ("EVAL", eval_file)]:
    with open(fname) as f:
        lines = f.readlines()
    first = json.loads(lines[0])
    print(f"\n\u2705 {label}: {fname}")
    print(f"   Esempi: {len(lines):,}")
    print(f"   Roles: {[m['role'] for m in first['messages']]}")
    print(f"   Preview: {first['messages'][-1]['content'][:120]}...")

In [ ]:
# Cell 5: Preparazione dataset
from datasets import load_dataset

train_dataset = load_dataset("json", data_files=train_file, split="train")
eval_dataset  = load_dataset("json", data_files=eval_file, split="train")

def format_chatml(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_dataset = train_dataset.map(format_chatml, num_proc=4)
eval_dataset  = eval_dataset.map(format_chatml, num_proc=4)

print(f"\u2705 Dataset pronti!")
print(f"   Train: {len(train_dataset):,} | Eval: {len(eval_dataset):,}")
print(f"\n\ud83d\udcdd Esempio formattato (primi 500 char):")
print(train_dataset[0]["text"][:500])

In [ ]:
# Cell 6: Training con SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 4,
    packing = True,
    args = TrainingArguments(
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 2,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        lr_scheduler_type = "cosine",
        warmup_ratio = 0.05,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        eval_strategy = "steps",
        eval_steps = 200,
        save_strategy = "steps",
        save_steps = 200,
        save_total_limit = 3,
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss",
        logging_steps = 25,
        report_to = "none",
        output_dir = "galileo-brain-v6",
        seed = 3407,
    ),
)

total_samples = len(train_dataset)
effective_batch = 8 * 2
steps_per_epoch = total_samples // effective_batch
total_steps = steps_per_epoch * 3
print(f"\u2705 Trainer configurato!")
print(f"   Effective batch size: {effective_batch}")
print(f"   Steps/epoch: ~{steps_per_epoch} | Total: ~{total_steps}")
print(f"   bf16: {is_bfloat16_supported()} | Packing: ON")
print(f"\n\ud83d\ude80 Esegui Cell 7 per avviare il training!")

In [ ]:
# Cell 7: TRAINING
import time

print("\ud83d\ude80 Training v6 avviato...")
print(f"   Dataset: 20,319 esempi (4 strati)")
print(f"   Stima: ~90 min su A100\n")

start = time.time()
stats = trainer.train()
elapsed = time.time() - start

print(f"\n{'='*50}")
print(f"\u2705 TRAINING v6 COMPLETATO!")
print(f"   Tempo: {elapsed/60:.1f} min")
print(f"   Loss finale: {stats.training_loss:.4f}")
print(f"   Steps totali: {stats.global_step}")
print(f"   VRAM picco: {torch.cuda.max_memory_allocated()/1024**3:.1f} GB")
print(f"{'='*50}")

In [ ]:
# Cell 8: Test inferenza (42 azioni + context + adversarial)
from unsloth import FastLanguageModel
import json

FastLanguageModel.for_inference(model)

test_cases = [
    # === SIMULATION CONTROL ===
    ("avvia la simulazione", "action"),
    ("ferma tutto", "action"),
    ("resetta il circuito", "action"),
    ("cancella tutto dalla breadboard", "action"),
    ("annulla l'ultima cosa", "action"),
    ("rifai quello che ho tolto", "action"),
    # === COMPILE & EDITOR ===
    ("compila il codice", "action"),
    ("apri l'editor", "action"),
    ("chiudi il pannello codice", "action"),
    ("passa a Scratch", "action"),
    ("torna ad Arduino C++", "action"),
    ("rimetti il codice originale", "action"),
    # === NAVIGATION ===
    ("carica l'esperimento del semaforo", "navigation"),
    ("apri il manuale", "navigation"),
    ("vai avanti col montaggio", "action"),
    ("torna al passo precedente", "action"),
    # === CIRCUIT ===
    ("metti un LED rosso sulla breadboard", "circuit"),
    ("aggiungi un buzzer e un pulsante", "circuit"),
    ("togli il resistore", "circuit"),
    ("collega il LED al pin 13", "circuit"),
    ("evidenzia il condensatore", "circuit"),
    # === INTERACTION ===
    ("premi il pulsante", "circuit"),
    ("misura la tensione sul LED", "circuit"),
    ("fai la diagnosi del circuito", "action"),
    ("cosa c'e' sul circuito?", "action"),
    # === UI ===
    ("mostra la lista componenti", "action"),
    ("apri il monitor seriale", "action"),
    ("fammi vedere le scorciatoie", "action"),
    # === EDUCATIONAL ===
    ("fammi un quiz", "action"),
    ("cerca un video sulle resistenze", "action"),
    ("crea un appunto su oggi", "action"),
    # === CODE (needs_llm=true) ===
    ("scrivi il codice per far lampeggiare un LED", "code"),
    ("come faccio a leggere il sensore di luce?", "code"),
    # === TUTOR (needs_llm=true) ===
    ("come funziona un condensatore?", "tutor"),
    ("cos'e' la legge di Ohm?", "tutor"),
    # === VISION ===
    ("guarda il mio circuito", "vision"),
    ("che errore c'e' nel mio circuito?", "vision"),
    # === TEACHER ===
    ("mostra i risultati della classe", "teacher"),
    # === ADVERSARIAL ===
    ("daje fallo anda'", "action"),          # dialect
    ("fai il compile please", "action"),     # mixed lang
    ("avvía la simulazzione", "action"),     # heavy typo
    ("cosa succede se avvio?", "tutor"),     # TRAP!
    ("fai quella cosa del circuito", "action"),  # vague
    ("non toccare niente!", "action"),       # negation
]

system_prompt = open(train_file).readline()
system_prompt = json.loads(system_prompt)['messages'][0]['content']

passed = 0
for user_msg, expected_intent in test_cases:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_msg},
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    output = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.1, do_sample=True)
    response = tokenizer.decode(output[0][inputs.shape[-1]:], skip_special_tokens=True).strip()
    
    if "</think>" in response:
        response = response.split("</think>")[-1].strip()
    
    try:
        parsed = json.loads(response)
        intent = parsed.get("intent", "???")
        ok = "\u2705" if intent == expected_intent else "\u274c"
        if intent == expected_intent:
            passed += 1
        print(f"{ok} [{intent:10}] {user_msg[:55]}")
    except json.JSONDecodeError:
        print(f"\u274c [PARSE ERR] {user_msg[:55]}")
        print(f"   Raw: {response[:100]}")

print(f"\n{'='*50}")
print(f"\ud83d\udcca Risultato: {passed}/{len(test_cases)} ({100*passed/len(test_cases):.1f}%)")
print(f"   Target: >=90% ({int(len(test_cases)*0.9)}/{len(test_cases)})")

In [ ]:
# Cell 9: Eval suite completa (108 esempi)
import json

eval_results = {"correct": 0, "wrong": 0, "parse_error": 0, "by_intent": {}}

with open(eval_file) as f:
    eval_examples = [json.loads(line) for line in f]

print(f"\ud83e\uddea Eval su {len(eval_examples)} esempi...\n")

for i, example in enumerate(eval_examples):
    messages = example["messages"][:2]
    expected = json.loads(example["messages"][2]["content"])
    expected_intent = expected["intent"]
    
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    output = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.1, do_sample=True)
    response = tokenizer.decode(output[0][inputs.shape[-1]:], skip_special_tokens=True).strip()
    
    if "</think>" in response:
        response = response.split("</think>")[-1].strip()
    
    if expected_intent not in eval_results["by_intent"]:
        eval_results["by_intent"][expected_intent] = {"correct": 0, "total": 0}
    eval_results["by_intent"][expected_intent]["total"] += 1
    
    try:
        parsed = json.loads(response)
        if parsed.get("intent") == expected_intent:
            eval_results["correct"] += 1
            eval_results["by_intent"][expected_intent]["correct"] += 1
        else:
            eval_results["wrong"] += 1
            print(f"  \u274c [{i}] expected={expected_intent} got={parsed.get('intent')} | {messages[1]['content'][:60]}")
    except json.JSONDecodeError:
        eval_results["parse_error"] += 1
        print(f"  \ud83d\udca5 [{i}] PARSE ERROR | {response[:80]}")
    
    if (i+1) % 25 == 0:
        print(f"  ... {i+1}/{len(eval_examples)} done")

total = eval_results["correct"] + eval_results["wrong"] + eval_results["parse_error"]
acc = eval_results["correct"] / total * 100

print(f"\n{'='*50}")
print(f"\ud83d\udcca EVAL RESULTS v6")
print(f"   Intent accuracy: {eval_results['correct']}/{total} ({acc:.1f}%)")
print(f"   Wrong intent:    {eval_results['wrong']}")
print(f"   Parse errors:    {eval_results['parse_error']}")
print(f"\n\ud83d\udccb Per-intent breakdown:")
for intent, data in sorted(eval_results["by_intent"].items()):
    pct = data['correct'] / data['total'] * 100 if data['total'] > 0 else 0
    bar = "\u2588" * int(pct/5) + "\u2591" * (20 - int(pct/5))
    print(f"   {intent:12} {bar} {data['correct']:3}/{data['total']:3} ({pct:.0f}%)")
print(f"\n   Target: >=95% intent accuracy")

In [ ]:
# Cell 10: Export GGUF q4_k_m
print("\ud83d\udce6 Esportazione GGUF q4_k_m...")
print("   Questo richiede ~5-10 minuti\n")

model.save_pretrained_gguf(
    "galileo-brain-v6-gguf",
    tokenizer,
    quantization_method = "q4_k_m",
)

print("\n\u2705 GGUF esportato!")

import os, glob as g
gguf_dirs = g.glob("galileo-brain-v6-gguf*")
for d in gguf_dirs:
    if os.path.isdir(d):
        for f in sorted(os.listdir(d)):
            size_mb = os.path.getsize(os.path.join(d, f)) / 1024 / 1024
            if size_mb > 1:
                print(f"   {d}/{f}: {size_mb:.1f} MB")

In [ ]:
# Cell 11: Download GGUF + Modelfile
import os, glob as g
from google.colab import files

# Trova il file GGUF
gguf_files = g.glob("galileo-brain-v6-gguf*/*.gguf")
modelfiles = g.glob("galileo-brain-v6-gguf*/Modelfile")

if gguf_files:
    gguf_path = gguf_files[0]
    size_gb = os.path.getsize(gguf_path) / 1024**3
    print(f"\u2705 GGUF: {gguf_path} ({size_gb:.2f} GB)")
    print(f"\n\u2b07\ufe0f  Scaricando GGUF...")
    files.download(gguf_path)

if modelfiles:
    print(f"\n\u2b07\ufe0f  Scaricando Modelfile...")
    files.download(modelfiles[0])
    with open(modelfiles[0]) as f:
        print(f"\n\ud83d\udcc4 Modelfile:")
        print(f.read()[:500])

print("\n" + "="*50)
print("\ud83c\udf89 TRAINING v6 COMPLETO!")
print("   Prossimo passo: ollama create galileo-brain-v6 -f Modelfile")